# Logs CPF Intergrall VF1 — Gold Layer

**Migrated from:** Alteryx workflow `Logs CPF Intergrall VF1.yxmd`
**Migration date:** 2026-08-06
**Medallion tier:** Gold (indicador de negócio)
**Source tables:** `{silver}.log_itgl_consultas`, `{silver}.rh_funcionarios`
**Target:** `{gold}.logs_cpf_intergrall` + `.xlsx` datado no Volume

## Alteryx Tool Mapping
| Step | Alteryx Tool | ToolID | Databricks Operation |
|---|---|---|---|
| 1 | Join → saída **Left** | 18 | SQL `LEFT ANTI JOIN` |
| 2 | Formula (`FULLPATCH`, `AssuntoEmail`) | 19, 27 | nomes montados em `00_config` |
| 3 | Select | 21, 31, 32, 33 | SQL `SELECT` explícito |
| 4 | BlockUntilDone | 20 | ordem sequencial das células |
| 5 | PortfolioComposerTable | 22 | `.toPandas()` → `.xlsx` (Python) |
| 6 | TextInput (`"Sem registro"`) | 28 | branch de resultado vazio |
| 7 | AppendFields | 29 | dispensado (ver nota) |
| 8 | Filter (`IsEmpty`) | 30 | `if n_resultado == 0` |
| 9 | Output Data (.xlsx) | 10, 34 | `.xlsx` no Volume + tabela Delta |
| 10 | Email | 1 | notificação do Lakeflow Job (stub) |

## Por que é Gold e não Silver
A saída é row-level, não agregada — mas é o **indicador de auditoria** que
alimentava o e-mail e o relatório da Auditoria Contínua, ou seja, o artefato
de consumo do negócio. O `LEFT ANTI JOIN` que a produz é o cálculo do
indicador, não uma limpeza.

In [0]:
%run ./00_config

In [0]:
# Imports mantidos para o export .xlsx (célula posterior)
import os
import pandas as pd
import tempfile
from pyspark.sql import functions as F

## 1. Criar tabela Gold + Anti-join (Tool 18, saída `Left`)

O Tool 18 está ligado pela saída **`Left`** (`.yxmd:853-856`) — os registros do
log que **não** casaram. Em SQL: `LEFT ANTI JOIN`.

A célula seguinte cria a tabela (se não existe) e a posterior insere os
resultados do anti-join com `REPLACE WHERE` para idempotência diária.

A chave é `log_itgl_usu = nome_funcionario`, comparação exata de texto livre.
Ver a nota de revisão no fim do notebook.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `${catalog}`.${gold_schema}.logs_cpf_intergrall (
  data_execucao    DATE      COMMENT 'Data da execução do indicador',
  log_itgl_seq     BIGINT    COMMENT 'Sequencial do log Intergrall (origem Sybase)',
  log_itgl_usu     STRING    COMMENT 'Usuário que efetuou a consulta — chave do anti-join contra o RH',
  log_itgl_nm      STRING    COMMENT 'Nome do usuário',
  log_itgl_nm_cns  STRING    COMMENT 'Nome do cliente consultado',
  log_itgl_cpf_cns STRING    COMMENT 'CPF consultado',
  log_itgl_dat     STRING    COMMENT 'Data da consulta (dd-MM-yyyy, / trocada por -)',
  log_itgl_hor     STRING    COMMENT 'Hora da consulta',
  log_itgl_obs     STRING,
  log_itgl_arq     STRING,
  log_itgl_dat_itg TIMESTAMP,
  log_itgl_usu_itg STRING,
  log_itgl_ati     INT,
  chave_cons       STRING    COMMENT 'usu + nm + nm_cns + cpf_cns + dat — chave de deteção de repetição'
)
USING DELTA
CLUSTER BY (data_execucao, log_itgl_usu, log_itgl_cpf_cns)
COMMENT 'Gold — indicador de auditoria contínua: consultas REPETIDAS de CPF no Intergrall feitas por usuários FORA dos cargos de direção. Migrado de Logs CPF Intergrall VF1.yxmd.'

## 2. Anti-join + escrita na Gold (histórico completo)

A célula seguinte executa o anti-join e grava na tabela Delta usando
`REPLACE WHERE` dinâmico sobre o intervalo de datas presente em Silver.

* **`data_execucao`** = data real do log (`data_convertida`), não a data da execução.
* **Idempotência**: o REPLACE WHERE substitui apenas as datas reprocessadas.
* **Histórico**: meses anteriores permanecem intactos; a cada run diário,
  o mês corrente é recalculado.

In [0]:
# Dynamic REPLACE WHERE based on actual date range in Silver
# Gold accumulates full history — each run replaces only the date window present in Silver
date_bounds = spark.sql(f"""
  SELECT MIN(data_convertida) AS min_dt, MAX(data_convertida) AS max_dt
  FROM {T_SILVER_LOG}
""").collect()[0]

if date_bounds.min_dt is None:
    print("Silver log vazio — nada a inserir na Gold")
else:
    replace_min = date_bounds.min_dt.isoformat()
    replace_max = date_bounds.max_dt.isoformat()

    spark.sql(f"""
      INSERT INTO {T_GOLD}
      REPLACE WHERE data_execucao >= '{replace_min}' AND data_execucao <= '{replace_max}'
      SELECT
        l.data_convertida    AS data_execucao,
        l.log_itgl_seq,
        l.log_itgl_usu,
        l.log_itgl_nm,
        l.log_itgl_nm_cns,
        l.log_itgl_cpf_cns,
        l.log_itgl_dat,
        l.log_itgl_hor,
        l.log_itgl_obs,
        l.log_itgl_arq,
        l.log_itgl_dat_itg,
        l.log_itgl_usu_itg,
        l.log_itgl_ati,
        l.chave_cons
      FROM {T_SILVER_LOG} l
      LEFT ANTI JOIN {T_SILVER_RH} r
        ON l.log_itgl_usu = r.nome_funcionario
    """)
    print(f"Gold atualizada: data_execucao de {replace_min} a {replace_max}")

## 3. Contagem e export diário

Lê da Gold com filtro `>= ontem` (replicando o Alteryx Tool 25) para o
export `.xlsx` diário. A Gold mantém histórico completo; o arquivo é a
"janela do dia" que a auditoria recebe.

In [0]:
# Lê da Gold com filtro dos últimos X dias (adapta Alteryx Tool 25)
# Gold acumula histórico completo; o export pega só a janela recente
# DIAS_EXPORT vem de 00_config (widget "dias_export", default=1, configurável pelo job)
data_corte = (HOJE - __import__('datetime').timedelta(days=DIAS_EXPORT)).isoformat()
resultado = spark.table(T_GOLD).filter(f"to_date(log_itgl_dat, 'dd-MM-yyyy') >= '{data_corte}'")
n_resultado = resultado.count()
print(f"Export: {n_resultado:,} registros com log_itgl_dat >= {data_corte} (últimos {DIAS_EXPORT} dias)")

## 4. `.xlsx` no Volume de destino (Tools 10, 22, 28, 34)

Os Outputs 10 e 34 gravavam no **mesmo** caminho `FULLPATCH` e eram mutuamente
exclusivos: o 34 só disparava com resultado vazio (o `Filter IsEmpty` do Tool 30
sobre o Append do texto `"Sem registro"`). Aqui é um `if` explícito, o que
dispensa o `crossJoin` do AppendFields (Tool 29) — ele só existia para levar o
literal `"Sem registro"` adiante num fluxo sem condicionais.

O `.xlsx` exige `toPandas()`, então o resultado passa pelo driver — daí o teto
de segurança. O indicador é diário e filtrado, então o volume é pequeno; se o
limite estourar, o dado íntegro continua na tabela Delta.

In [0]:
MAX_LINHAS_XLSX = 500_000
CABECALHO_XLSX = {"chave_cons": "Chave cons."}

# Create output directory on Volume using dbutils (os.makedirs can fail on FUSE)
dbutils.fs.mkdirs(OUTPUT_DIR.replace("/Volumes/", "dbfs:/Volumes/"))

# Filter xlsx export to last DIAS_EXPORT days (even in full mode)
data_corte = (HOJE - __import__('datetime').timedelta(days=DIAS_EXPORT)).isoformat()
resultado = resultado.filter(f"to_date(log_itgl_dat, 'dd-MM-yyyy') >= '{data_corte}'")
n_resultado = resultado.count()

if n_resultado == 0:
    # Tools 28/29/30/34: arquivo-marcador de "nada a reportar"
    pdf = pd.DataFrame({"LOG": ["Sem registro"]})
    print("resultado vazio — gravando arquivo 'Sem registro'")
else:
    if n_resultado > MAX_LINHAS_XLSX:
        raise ValueError(
            f"{n_resultado:,} registros excedem o limite de {MAX_LINHAS_XLSX:,} para o "
            f".xlsx. Os dados estão íntegros em {T_GOLD}; investigue o volume antes de "
            "gerar o arquivo."
        )
    pdf = resultado.toPandas().rename(columns=CABECALHO_XLSX)

# Write to local temp file, then sequential binary copy to Volume
# (openpyxl uses random-access ZIP I/O that fails on UC Volumes FUSE)
with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tmp:
    tmp_path = tmp.name

try:
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        pdf.to_excel(writer, sheet_name=SHEET_NAME, index=False)
    # Sequential binary copy — FUSE handles this reliably
    with open(tmp_path, "rb") as src, open(OUTPUT_XLSX, "wb") as dst:
        dst.write(src.read())
finally:
    os.unlink(tmp_path)

print(f"gravado {OUTPUT_XLSX} ({len(pdf):,} linhas)")

## 5. Notificação — a configurar no Job (Tool 1 + evento AfterError)

O Alteryx enviava dois e-mails via `smtp.sendgrid.net` com a API key do SendGrid
**encriptada dentro do `.yxmd`** (linhas 55 e 970):

1. Tool 1 — resultado, assunto `AssuntoEmail`, anexo `FULLPATCH2`.
2. Evento `AfterError` — `ERRO NA EXECUÇÃO: LOGS INTERGRALL`.

Nenhuma credencial foi trazida. Configure no Lakeflow Job:

* `email_notifications.on_success` → `auditoria.continua@bancobmg.com.br`
* `email_notifications.on_failure` → idem (substitui o `AfterError`)

O anexo não tem equivalente direto: a notificação do Job leva o link da
execução. O arquivo fica no Volume, e o indicador pode ser lido pela tabela
Delta ou por um dashboard AI/BI sobre ela — o que remove a necessidade do anexo.

In [0]:
print(f"Assunto original : {ASSUNTO_EMAIL}")
print(f"Anexo original   : {OUTPUT_XLSX}")
print(f"Destinatário     : {EMAIL_TO}")

## 6. Cleanup

Com a lógica em SQL puro, não há tabelas temporárias a limpar — o resultado
já está persistido na tabela Gold. O notebook 04 lê diretamente de lá.

In [0]:
dbutils.notebook.exit(
    f"GOLD OK | {n_resultado} registros | {T_GOLD} | {OUTPUT_XLSX}"
)

## Pontos para validar com a auditoria antes de promover

1. **A direção do anti-join.** O fluxo guarda consultas repetidas feitas por
   usuários que **não** são diretores/presidência. Para um indicador de
   "Monitoramento CPF Intergrall" é plausível (consultas fora da alçada
   executiva), mas é o inverso do que a documentação de análise descrevia, e a
   intenção foi inferida da ligação dos nós. Se o alvo é vigiar a própria
   direção, `how="left_anti"` deve virar `how="inner"`.
2. **Join por nome completo.** `log_itgl_usu = nome_funcionario` é comparação
   exata de texto livre: acento, abreviação, espaço duplo ou caixa diferente já
   fazem o registro escapar do join — e, num anti-join, **escapar significa
   entrar no resultado**. Falsos positivos são o modo de falha esperado.
   `cpf_ret` já existe no RH e o log tem `log_itgl_cpf_cns`; cruzar por CPF
   seria bem mais robusto, e talvez fosse a intenção do `CPF_RET` que ficou sem
   uso no original.
3. **Duas janelas de tempo empilhadas.** O log é lido do mês corrente, mas o
   recorte final mantém só a partir de ontem. A `Chave cons.` é avaliada sobre o
   mês todo, então uma consulta de ontem é marcada como repetida se houve outra
   igual em qualquer dia do mês — mas, no dia 1º, o histórico do mês anterior
   desaparece e a repetição deixa de ser detectada. Com a tabela Delta é fácil
   trocar por uma janela móvel de 30 dias, se a borda não for intencional.